# What our TinyStories budget actually purchased

Chapter 6 · Day 9

The model consumed 48.84M valid targets while processing 229.38M padded positions. Compare measured clocks and loss from the real archived run; avoid loading weights or starting a server.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/dongxi_llms').exists())
sys.path.insert(0, str(ROOT / 'src'))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(909)
plt.rcParams.update({'figure.figsize': (8, 3.6), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': False})

In [ ]:
from dongxi_llms.course_foundations import budget_summary
e = json.loads((ROOT / 'experiments/reports/2026-09-14-tinystories-learning-result.json').read_text())
s = budget_summary(e)
print(json.dumps(s, indent=2))
assert e['completion']['completed_updates'] == 14000
assert abs(s['valid_fraction'] - .2129253932) < 1e-8

Prediction: why do update-timer throughput and total-run throughput differ when they count the same valid labels? Reference: the elapsed-time scopes differ. Preparation is outside both. The ratio of sums weights longer updates correctly; a mean of heterogeneous per-batch ratios answers another question.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
axes[0].bar(['Processed', 'Valid', 'Prepared'], [e['metrics_summary']['processed_positions']/1e6, e['metrics_summary']['valid_targets']/1e6, e['data_manifest']['train']['valid_targets']/1e6], color=['#adb4bd','#3265ad','#a56928'])
axes[0].set(ylabel='Positions / target presentations (millions)', title='Different counts, different questions')
axes[1].plot([r['update'] for r in e['validation_curve']], [r['loss'] for r in e['validation_curve']], color='#3265ad')
axes[1].set(xlabel='Completed updates', ylabel='Fixed-development NLL (nats)')
fig.tight_layout()
plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![01 read training clocks: reference plot 01](../figures/chapter-06/day-09-01_read_training_clocks-01.png)

This fixed image comes from the CPU reference or the explicitly identified saved evidence; changing inputs does not update the preview automatically.

Reference explanation: prepared targets describe availability; consumed labels describe exposure; processed positions include padding. They are not three estimates of unique linguistic knowledge. The two panels use actual stored counts and loss, not extrapolated training progress.

Intervention: calculate token rate if someone incorrectly divides processed positions by total runtime. Reference: it becomes a larger positions/s figure, but reporting it as valid targets/s is a unit error. Changing the denominator can create the appearance of faster learning without changing a single update.

What remains: a systems change such as length-aware batching needs an actual controlled run; the archive alone cannot measure its speedup. See the next notebook for attention-boundary invariants that such a change must retain.